In [28]:
import pandas as pd
import json

def extract_conversations(csv_file):
    # Read the CSV file
    df = pd.read_csv(csv_file, sep='\t')
    
    # Dictionary to store all conversations
    all_conversations = []
    
    # Group by dialog_id to process each conversation
    for dialog_id in df['dialog_id'].unique():
        dialog_df = df[df['dialog_id'] == dialog_id].sort_values('utt_id')
        movie_id = df[df['dialog_id'] == dialog_id]['movie_id'].iloc[0]
        print(movie_id)
        
        conversation = []
        current_speaker = None
        current_content = []
        
        for _, row in dialog_df.iterrows():
            speaker = row['speaker']
            text = row['text']
            
            # If same speaker continues, accumulate text
            if speaker == current_speaker:
                current_content.append(text)
            else:
                # If we have accumulated content, add it to conversation
                if current_content and current_speaker is not None:
                    prev_role = 'user' if current_speaker == 'SEEKER' else 'assistant'
                    conversation.append({
                        'role': prev_role,
                        'content': ' '.join(current_content)
                    })
                
                # Start new speaker's content
                current_speaker = speaker
                current_content = [text]
        
        # Don't forget the last accumulated content
        if current_content and current_speaker is not None:
            role = 'user' if current_speaker == 'SEEKER' else 'assistant'
            conversation.append({
                'role': role,
                'content': ' '.join(current_content)
            })

        # Add conversation to the list if it has content
        if conversation:
            all_conversations.append({
                'dialog_id': dialog_id,
                'ground_truth': movie_id,
                'conversation': conversation
            })
    
    return all_conversations

def save_conversations_to_json(conversations, output_file):
    # Save to JSON file
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(conversations, f, indent=2, ensure_ascii=False)

# Main execution
if __name__ == "__main__":
    # Extract conversations from CSV
    conversations = extract_conversations('raw/train.tsv')
    
    # Save to JSON file
    save_conversations_to_json(conversations, 'multiturn_form/train.json')
    
    print(f"Extracted {len(conversations)} conversations and saved to conversations.json")
    
    # Print a sample conversation for verification
    if conversations:
        print("\nSample conversation:")
        sample_conv = conversations[0]['conversation']
        for i, turn in enumerate(sample_conv[:4]):  # Show first 4 turns
            print(f"{turn['role']}: {turn['content']}")

sE6fTeuKZkg
iyd0eYFl4eY
2T3fHtay4eg
ZUCe7raZHAc
BXkSost_0Nc
JaNs468a59c
Budwd_uAnsk
KbmK9o9DJn4
jvQUIt0BWcU
43oUoc7kEMk
Budwd_uAnsk
lG7DGMgfOb8
zwhP5b4tD6g
LUG2U-IxPx0
d4IXv5_E33M
iyd0eYFl4eY
rgXplohCw5o
_EpH1zU-NIU
IfvzEXhhWNk
F7Ug863S8dQ
XQw658a-8UQ
pMYyemI0nxY
KzJNYYkkhzc
zvcKNaWgpXA
IJPWP3udqE8
OiRQOpC0nhg
Qt2AyS-Zinw
BXkSost_0Nc
iyd0eYFl4eY
dLaBxTeRHV4
znyWYYHxpDM
ogLaoLFWWE4
YHziuA9bM7k
prH-hf9s9_w
dMaq_pfxs-0
_EpH1zU-NIU
KBiOF3y1W0Y
79uuJdcC8wQ
AuzyODgWRp4
zvcKNaWgpXA
HSaynNIu5vA
ee1172yeqyE
woHTUsl66BY
r7pGnfDvlWg
x3guLR8aBpU
BoohRoVA9WQ
-AKU-T7_rDA
41dUp3kK-0w
hcn0T5Zygnk
N0io2w_6vT8
JaNs468a59c
Vo96qDiGUiI
84TouqfIsiI
vtejTSKBT1k
nY9F7DMOoGM
r_d6RK6o9t0
_DSGAeeDXO0
JaNs468a59c
ie365Vzyef8
MneQ4hAwsBI
-gieJQejbHQ
rgXplohCw5o
jTSlHYxAnEs
jAI7rF0eQyQ
fNeCB2ALNoA
3W0teWwq5wg
iyd0eYFl4eY
8sLrLtH-nJ4
GPKqQR64u_Q
0fUCuvNlOCg
BXkSost_0Nc
h-k0-Z9b4Cw
70UY9X7qhE0
NAq6vubHdCg
9rQu77pgnpg
XQw658a-8UQ
vCkImyTNZGo
hhSpF4v_SBY
dCz30g0azGk
mV9kxSK_-0I
Cgnk3MLw9TM
KWs5TTn19qk
pPhISgw3I2w
3iTm

In [3]:
import os

# Get the current working directory
current_directory = os.getcwd()

# Print the current working directory
print(current_directory)

/home/sagemaker-user/csbai/multiturn_rl


In [29]:
import pandas as pd
import json
import csv

def format_conversation_simple(conversation):
    """
    Alternative: Simple format with role indicators
    """
    formatted_parts = []
    
    for turn in conversation:
        role = turn.get('role', '')
        content = turn.get('content', '')
        formatted_parts.append(f"[{role.upper()}] {content}")
    
    return ' | '.join(formatted_parts)

def json_to_csv_simple(json_file, csv_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    csv_data = []
    for entry in data:
        dialog_id = entry.get('dialog_id', '')
        conversation = entry.get('conversation', [])
        ground_truth = entry.get('ground_truth', '')
        
        # # Use simple format
        # conversation_str = format_conversation_simple(conversation)
        
        csv_data.append({
            'dialog_id': dialog_id,
            'ground_truth': ground_truth,
            'conversation': conversation
        })
    
    df = pd.DataFrame(csv_data)
    df.to_csv(csv_file, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)

json_to_csv_simple('multiturn_form/train.json', 'multiturn_form/train.csv')

In [10]:
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk

ds_dict = load_dataset("csv", data_files="/home/sagemaker-user/csbai/multiturn_rl/datasets/inspired/multiturn_form/train.csv")



In [19]:
print(ds_dict['train']['conversation'])

[{'role': 'assistant', 'content': 'Hi There! What types of movies do you like to watch?'}, {'role': 'user', 'content': "Hello! I'm more of an action movie or a good romance and mystery movie."}, {'role': 'assistant', 'content': 'I just saw the trailer for Knives Out when I went to see Joker and it looked like a good mix of action and mystery!'}, {'role': 'user', 'content': 'I seen that one too as I seen Joker about a month ago. I thought about asking my fiance about going and seeing it.'}, {'role': 'assistant', 'content': 'It looks like a good movie for people who like many different movies. It also has a great cast! I was surprised to see Chris Evans in the trailer!'}, {'role': 'user', 'content': "Maybe with Chris Evans in it it'll be easier to convince my fiance to see it. Do you know who else is in the cast?"}, {'role': 'assistant', 'content': 'Daniel Craig and Jamie Lee Curtis are also in the cast. Daniel Craig does a lot of 007 so definitely a good hearthrob role to convince the m